<a href="https://colab.research.google.com/github/FC-Andrade/Analises-complementares/blob/main/TEMPORAL_CLUSTER_EVOLUTION_(Apo_vs_Complex).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
# TEMPORAL CLUSTER EVOLUTION -  (Apo vs Complex)
# =====================================================
# Requer: MDAnalysis, scikit-learn, numpy, matplotlib, seaborn

import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from MDAnalysis.analysis import align
from sklearn.cluster import DBSCAN

sns.set(style="whitegrid", context="talk")
plt.rcParams['figure.figsize'] = (12, 6)

# --- CONFIGURAÇÃO ---
selection = "protein and name CA"
stride = 10  # usar 1 para maior resolução temporal
eps = 0.25   # parâmetro DBSCAN
min_samples = 3

# =====================================================
# 1️⃣ Carregar e concatenar as trajetórias (3 réplicas)
# =====================================================
apo_files = [
    ("apo1.tpr", "apo1.xtc"),
    ("apo2.tpr", "apo2.xtc"),
    ("apo3.tpr", "apo3.xtc")
]
c16_files = [
    ("c16_1.tpr", "c16_1.xtc"),
    ("c16_2.tpr", "c16_2.xtc"),
    ("c16_3.tpr", "c16_3.xtc")
]

def concat_trajectories(file_list, selection):
    coords_all = []
    lengths = []
    for top, traj in file_list:
        u = mda.Universe(top, traj)
        atoms = u.select_atoms(selection)
        coords = np.array([atoms.positions.copy() for ts in u.trajectory[::stride]])
        coords_all.append(coords)
        lengths.append(len(coords))
    return np.concatenate(coords_all), lengths

coords_apo, len_apo = concat_trajectories(apo_files, selection)
coords_c16, len_c16 = concat_trajectories(c16_files, selection)

# =====================================================
# 2️⃣ Cálculo da matriz RMSD
# =====================================================
def rmsd_matrix(coords):
    n = coords.shape[0]
    mat = np.zeros((n, n))
    for i in range(n):
        diff = coords - coords[i]
        mat[i] = np.sqrt(np.mean(np.sum(diff**2, axis=2), axis=1))
    return mat

rmsd_all = {
    "Apo": rmsd_matrix(coords_apo),
    "C16": rmsd_matrix(coords_c16)
}

# =====================================================
# 3️⃣ Clustering (DBSCAN)
# =====================================================
def run_clustering(rmsd_mat, eps=0.25, min_samples=3):
    clustering = DBSCAN(eps=eps, min_samples=min_samples, metric="precomputed").fit(rmsd_mat)
    labels = clustering.labels_
    return labels

labels_apo = run_clustering(rmsd_all["Apo"], eps, min_samples)
labels_c16 = run_clustering(rmsd_all["C16"], eps, min_samples)

# =====================================================
# 4️⃣ Separar clusters por réplica
# =====================================================
def split_labels(labels, lengths):
    split = []
    start = 0
    for l in lengths:
        split.append(labels[start:start+l])
        start += l
    return split

split_apo = split_labels(labels_apo, len_apo)
split_c16 = split_labels(labels_c16, len_c16)

# =====================================================
# 5️⃣ Plot temporal dos clusters
# =====================================================
def plot_temporal_clusters(split_labels_list, color, title):
    fig, axs = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
    for i, labels in enumerate(split_labels_list):
        time = np.arange(len(labels)) * 100 / len(labels)  # normalizar para 100 ns
        axs[i].scatter(time, labels, s=15, color=color)
        axs[i].set_xlim(0, 100)
        axs[i].set_xlabel("Time (ns)")
        axs[i].set_title(f"Replica {i+1}")
    axs[0].set_ylabel("Cluster ID")
    plt.suptitle(title, color=color, fontsize=18)
    plt.tight_layout()
    plt.show()

plot_temporal_clusters(split_apo, "steelblue", "Apo")
plot_temporal_clusters(split_c16, "darkorange", "C16/S1")